# SYSE-822 Applied Aerodynamics, Project I
## Panel Method for Inviscid, Incompressible Flow over a 2D Airfoil

**Method in 5 steps (Hess-Smith panel method)**
1. Replace the airfoil by N straight panels.
2. Put a source of unknown strength on each panel and one common vortex strength on all panels.
3. Wall condition: no flow through the surface at the midpoint of every panel (N equations).
4. Kutta condition: flow leaves the sharp trailing edge smoothly (1 equation). This fixes the circulation, so it fixes lift.
5. Solve the (N+1) x (N+1) linear system, then get the surface speed, `Cp = 1 - (V/Vinf)^2`, and integrate Cp to get CL and CD.

Change the three inputs in the next cell and run all cells again.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.path import Path
from types import SimpleNamespace

CODE  = "2412"   # NACA 4-digit airfoil (try 0012, 4412, 6409 ...)
ALPHA = 5.0      # angle of attack in degrees
N     = 160      # number of panels

## 1. Geometry: NACA 4-digit airfoil and panels

In [ ]:
def camber(x, m, p):
    "Mean camber line yc(x) and its slope dyc/dx."
    if m == 0: return 0*x, 0*x
    yc  = np.where(x < p, m/p**2*(2*p*x - x**2), m/(1-p)**2*((1-2*p) + 2*p*x - x**2))
    dyc = np.where(x < p, 2*m/p**2*(p - x),      2*m/(1-p)**2*(p - x))
    return yc, dyc

def naca4(code, N):
    "Airfoil nodes ordered: trailing edge -> upper surface -> leading edge -> lower surface -> trailing edge."
    m, p, t = int(code[0])/100, int(code[1])/10, int(code[2:])/100
    x  = 0.5*(1 - np.cos(np.linspace(0, np.pi, N//2 + 1)))      # cosine spacing: more panels at the edges
    yt = 5*t*(0.2969*np.sqrt(x) - 0.1260*x - 0.3516*x**2 + 0.2843*x**3 - 0.1036*x**4)
    yc, dyc = camber(x, m, p); th = np.arctan(dyc)
    xu, yu = x - yt*np.sin(th), yc + yt*np.cos(th)
    xl, yl = x + yt*np.sin(th), yc - yt*np.cos(th)
    X, Y = np.r_[xu[::-1], xl[1:]], np.r_[yu[::-1], yl[1:]]
    X[-1], Y[-1] = X[0], Y[0]                                     # close the trailing edge
    return X, Y

def make_panels(code, N):
    x, y = naca4(code, N)
    dx, dy = np.diff(x), np.diff(y); l = np.hypot(dx, dy)
    tx, ty = dx/l, dy/l                                           # tangent of each panel
    return SimpleNamespace(x=x, y=y, l=l, tx=tx, ty=ty, nx=ty, ny=-tx,   # nx, ny = outward normal
                           xc=(x[:-1]+x[1:])/2, yc=(y[:-1]+y[1:])/2, N=len(l))

P = make_panels(CODE, N)
plt.figure(figsize=(9, 3))
plt.plot(P.x, P.y, "o-", ms=2, lw=1); plt.quiver(P.xc, P.yc, P.nx, P.ny, scale=25, color="r", width=0.002)
plt.axis("equal"); plt.title(f"NACA {CODE} with {P.N} panels (red = outward normals)"); plt.show()

## 2. Solver: build and solve the linear system
Each panel induces a velocity at any point. In the panel's own frame (xi along the panel, eta normal to it), with r1, r2 the distances to the two panel ends and D the angle the panel subtends:

* unit source: `u_xi = ln(r1^2/r2^2)/(4 pi)`, `u_eta = D/(2 pi)`
* unit vortex: `u_xi = -D/(2 pi)`, `u_eta = ln(r1^2/r2^2)/(4 pi)`

In [ ]:
def influence(P, px, py, on_wall=False):
    "Velocity (global x, y) at points px, py from a UNIT source panel and a UNIT vortex panel, for every panel."
    dx, dy = px[:, None] - P.x[None, :-1], py[:, None] - P.y[None, :-1]
    xi, eta = dx*P.tx + dy*P.ty, -dx*P.ty + dy*P.tx               # local panel coordinates
    ln = np.log((xi**2 + eta**2) / ((xi - P.l)**2 + eta**2))
    D  = np.arctan2(eta, xi - P.l) - np.arctan2(eta, xi)
    if on_wall:                                                    # a panel acting on its own midpoint
        i = np.arange(P.N); ln[i, i] = 0; D[i, i] = -np.pi
    glob = lambda a, b: (a*P.tx - b*P.ty, a*P.ty + b*P.tx)         # local -> global
    return glob(ln/(4*np.pi), D/(2*np.pi)), glob(-D/(2*np.pi), ln/(4*np.pi))

def solve(P, alpha_deg):
    n, a = P.N, np.radians(alpha_deg)
    Vx, Vy = np.cos(a), np.sin(a)                                  # free stream (Vinf = 1)
    (sx, sy), (vx, vy) = influence(P, P.xc, P.yc, on_wall=True)
    nx, ny, tx, ty = P.nx[:, None], P.ny[:, None], P.tx[:, None], P.ty[:, None]
    An, At = sx*nx + sy*ny, sx*tx + sy*ty                          # source effect: normal, tangential
    Bn, Bt = (vx*nx + vy*ny).sum(1), (vx*tx + vy*ty).sum(1)        # vortex effect: normal, tangential
    M = np.zeros((n+1, n+1)); rhs = np.zeros(n+1)
    M[:n, :n], M[:n, n] = An, Bn                                   # wall condition on every panel
    rhs[:n] = -(Vx*P.nx + Vy*P.ny)
    M[n, :n], M[n, n] = At[0] + At[-1], Bt[0] + Bt[-1]             # Kutta condition: Vt(first) + Vt(last) = 0
    rhs[n] = -(Vx*(P.tx[0] + P.tx[-1]) + Vy*(P.ty[0] + P.ty[-1]))
    sol = np.linalg.solve(M, rhs); sigma, gamma = sol[:n], sol[n]
    vt = Vx*P.tx + Vy*P.ty + At @ sigma + gamma*Bt                 # surface speed
    cp = 1 - vt**2
    Fx, Fy = -np.sum(cp*P.nx*P.l), -np.sum(cp*P.ny*P.l)            # pressure force (chord = 1)
    CL = -Fx*np.sin(a) + Fy*np.cos(a); CD = Fx*np.cos(a) + Fy*np.sin(a)
    CL_kj = -2*np.sum(vt*P.l)                                      # Kutta-Joukowski check: CL = -2*circulation
    return SimpleNamespace(alpha=alpha_deg, sigma=sigma, gamma=gamma, vt=vt, cp=cp, CL=CL, CD=CD, CL_kj=CL_kj)

S = solve(P, ALPHA)
print(f"NACA {CODE}, alpha = {ALPHA} deg, N = {N} panels")
print(f"CL (pressure integration) = {S.CL:.4f}")
print(f"CL (Kutta-Joukowski check)= {S.CL_kj:.4f}   <- independent check, should match")
print(f"CD (pressure integration) = {S.CD:.5f}   <- ideal value is 0")

## 3. Pressure coefficient (Cp) graph

In [ ]:
up = np.arange(P.N) < P.N//2          # first half of the panels = upper surface
plt.figure(figsize=(8, 5))
plt.plot(P.xc[up],  S.cp[up],  "r.-", label="upper surface")
plt.plot(P.xc[~up], S.cp[~up], "b.-", label="lower surface")
plt.gca().invert_yaxis()              # aerodynamic convention: suction (negative Cp) plotted upward
plt.xlabel("x/c"); plt.ylabel("Cp"); plt.grid(alpha=0.3); plt.legend()
plt.title(f"NACA {CODE}, alpha = {ALPHA} deg, CL = {S.CL:.3f}"); plt.show()
print(f"Minimum Cp (suction peak) = {S.cp.min():.2f} at x/c = {P.xc[S.cp.argmin()]:.3f}")

## 4. Lift coefficient CL for different angles of attack

In [ ]:
alphas = np.arange(-6, 15, 2.0)
CLs = np.array([solve(P, a).CL for a in alphas])

# thin airfoil theory for comparison: CL = 2*pi*(alpha - alpha_L0)
m, p = int(CODE[0])/100, int(CODE[1])/10
th = np.linspace(0, np.pi, 2001); f = camber(0.5*(1 - np.cos(th)), m, p)[1]*(np.cos(th) - 1)
a_L0 = -np.sum((f[1:] + f[:-1])/2*np.diff(th))/np.pi              # zero-lift angle (rad)
CL_thin = 2*np.pi*(np.radians(alphas) - a_L0)

slope, b = np.polyfit(alphas, CLs, 1)
plt.figure(figsize=(8, 5))
plt.plot(alphas, CLs, "o-", label="panel method"); plt.plot(alphas, CL_thin, "--", label="thin airfoil theory")
plt.axhline(0, c="k", lw=0.6); plt.axvline(0, c="k", lw=0.6)
plt.xlabel("alpha (deg)"); plt.ylabel("CL"); plt.grid(alpha=0.3); plt.legend(); plt.title(f"NACA {CODE}: lift curve"); plt.show()
print(f"Lift slope  = {slope:.4f} per deg = {np.degrees(slope):.2f} per rad   (thin airfoil theory: 2*pi = 6.28)")
print(f"Zero-lift angle = {-b/slope:.2f} deg   (thin airfoil theory: {np.degrees(a_L0):.2f} deg)")
print("Note: inviscid theory has no stall, so CL keeps rising. Real airfoils stall when the boundary layer separates.")

## 5. Drag coefficient (CD) analysis
In ideal (inviscid, incompressible) flow the drag is exactly zero. This is **d'Alembert's paradox**. The small CD our code gives is numerical error, and it should shrink as we add panels. Real drag comes from viscosity (skin friction and separation), which potential flow cannot model.

In [ ]:
Ns = [20, 40, 80, 160, 320]
CDs = [solve(make_panels(CODE, n), ALPHA).CD for n in Ns]
CD_alpha = [solve(P, a).CD for a in alphas]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(Ns, np.abs(CDs), "o-"); ax[0].set_xlabel("number of panels N"); ax[0].set_ylabel("|CD|")
ax[0].set_title(f"CD error shrinks with N (alpha = {ALPHA} deg)"); ax[0].grid(alpha=0.3)
ax[1].plot(alphas, CD_alpha, "o-"); ax[1].axhline(0, c="r", ls="--", label="exact inviscid CD = 0")
ax[1].set_xlabel("alpha (deg)"); ax[1].set_ylabel("CD"); ax[1].legend(); ax[1].grid(alpha=0.3); ax[1].set_title("CD vs alpha")
plt.tight_layout(); plt.show()

# rough viscous estimate for perspective (empirical flat plate + thickness form factor, NOT potential flow)
Re, t = 2e6, int(CODE[2:])/100
CD_visc = 2*(0.455/np.log10(Re)**2.58)*(1 + 2*t + 60*t**4)
print(f"Panel method CD (N={P.N})        = {S.CD:.5f}")
print(f"Rough viscous CD estimate (Re=2e6) = {CD_visc:.5f}  <- real drag is far larger than the numerical residual")

## 6. Velocity and pressure field visualisations

In [ ]:
# Velocity anywhere = free stream + effect of all source panels + effect of the vortex sheet
def velocity(P, S, X, Y):
    (sx, sy), (vx, vy) = influence(P, X.ravel(), Y.ravel())
    a = np.radians(S.alpha)
    u = np.cos(a) + sx @ S.sigma + S.gamma*vx.sum(1)
    v = np.sin(a) + sy @ S.sigma + S.gamma*vy.sum(1)
    inside = Path(np.c_[P.x, P.y]).contains_points(np.c_[X.ravel(), Y.ravel()])   # hide points inside the airfoil
    return [np.ma.array(q.reshape(X.shape), mask=inside.reshape(X.shape)) for q in (u, v)]

X, Y = np.meshgrid(np.linspace(-0.5, 1.5, 220), np.linspace(-0.5, 0.5, 110))
U, V = velocity(P, S, X, Y)
speed = np.hypot(U, V); Cp = 1 - speed**2

In [ ]:
# (a) Velocity field: speed contours, streamlines and velocity vectors
plt.figure(figsize=(11, 5))
cf = plt.contourf(X, Y, speed, levels=np.linspace(0, 1.8, 37), cmap="viridis", extend="max")
plt.streamplot(X, Y, U, V, color="w", linewidth=0.6, density=1.6, arrowsize=0.6)
plt.quiver(X[::9, ::9], Y[::9, ::9], U[::9, ::9], V[::9, ::9], color="k", scale=40, width=0.002)
plt.fill(P.x, P.y, "k"); plt.colorbar(cf, label="|V| / Vinf"); plt.axis("equal")
plt.title(f"Velocity field, NACA {CODE}, alpha = {ALPHA} deg"); plt.xlabel("x/c"); plt.ylabel("y/c"); plt.show()

In [ ]:
# (b) Pressure field: Cp contours (red = high pressure, blue = suction)
plt.figure(figsize=(11, 5))
cf = plt.contourf(X, Y, Cp, levels=np.linspace(-2, 1, 31), cmap="RdBu_r", extend="both")
plt.contour(X, Y, Cp, levels=np.linspace(-2, 1, 16), colors="k", linewidths=0.3)
plt.fill(P.x, P.y, "k"); plt.colorbar(cf, label="Cp"); plt.axis("equal")
plt.title(f"Pressure coefficient field, NACA {CODE}, alpha = {ALPHA} deg"); plt.xlabel("x/c"); plt.ylabel("y/c"); plt.show()

## Viva cheat sheet

**Why panels?** Potential flow is linear (Laplace equation), so we can add simple solutions (free stream + sources + vortices) and just solve for their strengths.

**Why sources AND a vortex?** Sources alone make a closed body with zero lift. The vortex adds circulation, which creates lift.

**What is the Kutta condition and why is it needed?** Without it, there are infinite valid solutions (any circulation). Nature picks the one where flow leaves the sharp trailing edge smoothly. In the code it is the last row of the matrix: Vt(first panel) + Vt(last panel) = 0.

**Where is the boundary condition?** At each panel midpoint (control point) the normal velocity is zero. This is the first N rows of the matrix.

**How is Cp computed?** Bernoulli: `Cp = 1 - (V/Vinf)^2`, with V the tangential surface speed.

**How are CL and CD computed?** Pressure acts normal to the wall: `F = -Cp * n * dl`. Sum over panels, then rotate into lift (perpendicular to the free stream) and drag (parallel).

**Why is CD about 0?** d'Alembert's paradox: no viscosity means no drag. The tiny non-zero value is discretisation error and shrinks with more panels.

**How do you know the code is right?** (1) CL from pressure integration equals CL from Kutta-Joukowski (circulation). (2) Lift slope is about 2 pi per rad and zero-lift angle matches thin airfoil theory. (3) CD tends to 0 as N grows.

**Limits:** no viscosity, so no stall and no real drag; incompressible, so valid only at low Mach number.